# 4. SAR interpretation, explainability and reporting

Notebook 3 ended with a model that scores badly on a scaffold split. This
notebook asks *why*, and produces the report a regulator would read.

**Covers:** `qsarkit.sar`, `qsarkit.explainability`, `qsarkit.reporting`,
`qsarkit.chemistry`

In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
np.set_printoptions(precision=3, suppress=True)

SMILES = [
    "OC(=O)c1ccccc1", "OC(=O)c1ccc(C)cc1", "OC(=O)c1ccc(Cl)cc1",
    "OC(=O)c1ccc(Br)cc1", "OC(=O)c1ccc(OC)cc1", "OC(=O)c1ccc(N)cc1",
    "CC(=O)Nc1ccccc1", "CC(=O)Nc1ccc(C)cc1", "CC(=O)Nc1ccc(Cl)cc1",
    "CC(=O)Nc1ccc(F)cc1", "CC(=O)Nc1ccc(OC)cc1", "CC(=O)Nc1ccc(O)cc1",
    "NC(=O)c1ccncc1", "NC(=O)c1ccc(C)nc1", "NC(=O)c1ccc(Cl)nc1",
    "NC(=O)c1ccc(OC)nc1", "CNC(=O)c1ccncc1", "CCNC(=O)c1ccncc1",
    "c1ccc2[nH]cnc2c1", "Cc1ccc2[nH]cnc2c1", "Clc1ccc2[nH]cnc2c1",
    "COc1ccc2[nH]cnc2c1", "Cn1cnc2ccccc21", "CCn1cnc2ccccc21",
]
Y = np.array([
    5.10, 5.35, 7.80, 5.40, 5.05, 4.90,
    6.20, 6.45, 6.70, 6.55, 6.10, 6.05,
    7.10, 7.35, 7.55, 7.20, 7.05, 6.95,
    8.00, 8.25, 8.45, 8.10, 7.90, 7.85,
])
mols = [Chem.MolFromSmiles(s) for s in SMILES]

from qsarkit.representation import MorganFingerprint
fingerprint = MorganFingerprint(radius=2, n_bits=512)
X = fingerprint.transform(mols)
X.shape

(24, 512)

## Diagnose before you model

An activity cliff is a pair of near-identical structures with very different
activity. Cliffs are where QSAR fails *by construction*: any model resting
on a smooth similarity assumption must predict them wrong.

Running this first tells you whether a regression model can work at all.

In [2]:
from qsarkit.sar import activity_cliff_report

cliffs = activity_cliff_report(mols, Y, similarity_threshold=0.5)
print(f"compounds:  {cliffs['n_compounds']}")
print(f"pairs:      {cliffs['n_pairs']}")
print(f"cliffs:     {cliffs['n_cliffs']}  ({cliffs['cliff_ratio']:.1%} of pairs)")
print(f"max SALI:   {cliffs['max_sali']:.2f}")

compounds:  24
pairs:      276
cliffs:     4  (1.4% of pairs)
max SALI:   6.38


> The default threshold is 0.85, the figure usually quoted in the cliff
> literature — calibrated for drug-sized molecules with a large shared core.
> These are small molecules, where a single-atom change alters every atom
> environment within the fingerprint radius, so similarity runs far below
> intuition. **Threshold to your data, not to the paper.**

In [3]:
print("the cliffs themselves:")
for cliff in cliffs["top_cliffs"]:
    print(f"  {SMILES[cliff.index_a]:22} pIC50 {Y[cliff.index_a]:.2f}")
    print(f"  {SMILES[cliff.index_b]:22} pIC50 {Y[cliff.index_b]:.2f}")
    print(f"      similarity {cliff.similarity:.2f}, "
          f"delta {cliff.delta:.2f}, SALI {cliff.sali:.1f}\n")

the cliffs themselves:
  OC(=O)c1ccc(Cl)cc1     pIC50 7.80
  OC(=O)c1ccc(N)cc1      pIC50 4.90
      similarity 0.55, delta 2.90, SALI 6.4

  OC(=O)c1ccccc1         pIC50 5.10
  OC(=O)c1ccc(Cl)cc1     pIC50 7.80
      similarity 0.55, delta 2.70, SALI 6.0

  OC(=O)c1ccc(C)cc1      pIC50 5.35
  OC(=O)c1ccc(Cl)cc1     pIC50 7.80
      similarity 0.55, delta 2.45, SALI 5.4

  OC(=O)c1ccc(Cl)cc1     pIC50 7.80
  OC(=O)c1ccc(Br)cc1     pIC50 5.40
      similarity 0.55, delta 2.40, SALI 5.3



Every cliff involves compound 2, the 4-chlorobenzoic acid — an outlier at
pIC50 7.80 in a family otherwise sitting near 5.2. Whether that is real
chemistry or a data error is a question for the assay records, and it is
exactly the kind of thing curation should surface *before* modelling.

In [4]:
print("which substituent changes cause them:")
for transformation, count in sorted(cliffs["top_transformations"].items()):
    print(f"  {count}x  {transformation}")

which substituent changes cause them:
  1x  [1*]C>>[1*]Cl
  1x  [1*]Cl>>[1*]Br
  1x  [1*]Cl>>[1*]N


## SALI and the activity landscape

SALI ranks pairs by how steep the cliff is: activity difference over
structural distance.

In [5]:
from qsarkit.sar import SALIAnalyzer

sali = SALIAnalyzer().sali_matrix(mols, Y)
print("SALI matrix:", sali.shape, " max:", round(float(sali.max()), 2))

i, j = np.unravel_index(np.argmax(sali), sali.shape)
print(f"steepest pair: {SMILES[i]} / {SMILES[j]}")

SALI matrix: (24, 24)  max: 6.38
steepest pair: OC(=O)c1ccc(Cl)cc1 / OC(=O)c1ccc(N)cc1


In [6]:
from qsarkit.sar import ActivityLandscapePlotter

landscape = ActivityLandscapePlotter().sas_data(mols, Y)
print(landscape["quadrant"].value_counts().to_string())

quadrant
scaffold hop    158
nondescript     107
smooth SAR       11


The SAS map assigns every pair to a quadrant. "Smooth SAR" regions are what
a QSAR model can learn; "activity cliff" regions are what it cannot;
"scaffold hop" pairs — similar activity from dissimilar structures — are the
interesting ones for design.

Only three quadrants appear here, and no pair lands in "activity cliff":
with these small molecules almost nothing clears the structural-similarity
bar the quadrant assignment uses, which is the same threshold caveat as
above.

In [7]:
from qsarkit.sar import SARIAnalyzer

scores = SARIAnalyzer().analyze(mols, Y)
for key, value in scores.items():
    print(f"{key:16} {value:.3f}")

continuity       0.102
discontinuity    0.065
sari             0.481


SARI condenses the whole landscape into one number, separating the
continuous component (smooth SAR, learnable) from the discontinuous one
(cliffs, not learnable).

## Matched molecular pairs

MMPs formalize "change one thing and see what happens", which is how
medicinal chemistry is actually done.

In [8]:
from qsarkit.sar import MatchedMolecularPairs

pairs = MatchedMolecularPairs().find_pairs(mols)
print(f"{len(pairs)} matched pairs found\n")
for pair in pairs[:6]:
    print(" ", pair)

38 matched pairs found

  MatchedPair([1*]C(=O)O>>[1*]NC(C)=O)
  MatchedPair([1*]C(=O)O>>[1*]NC(C)=O)
  MatchedPair([1*]C>>[1*]Cl)
  MatchedPair([1*]C>>[1*]Br)
  MatchedPair([1*]C>>[1*]OC)
  MatchedPair([1*]C>>[1*]N)


Because the transformation is recorded as a rule rather than a pair of
structures, identical changes across different cores aggregate — which is
what turns a list of pairs into a transferable design rule.

In [9]:
from collections import Counter

deltas = {}
for pair in pairs:
    key = str(pair).split("(")[-1].rstrip(")")
    deltas.setdefault(key, []).append(Y[pair.index_b] - Y[pair.index_a])

rows = [(k, len(v), float(np.mean(v))) for k, v in deltas.items() if len(v) >= 2]
frame = pd.DataFrame(rows, columns=["transformation", "n", "mean delta pIC50"])
frame.sort_values("mean delta pIC50", ascending=False).head(8)

,transformation,n,mean delta pIC50
1,[1*]C>>[1*]Cl,4,0.7750
0,C)=O,4,0.5375
5,[1*]C>>[1*]CC,2,-0.0750
4,=O)NCC,2,-0.1250
2,[1*]C>>[1*]OC,4,-0.2375
3,[1*]Cl>>[1*]OC,4,-1.0125


## Explaining a fitted model

Now the model itself. Which parts of a molecule is it keying on?

In [10]:
from qsarkit.models import QSARRegressor

model = QSARRegressor("rf", random_state=0).fit(X, Y)
print("training R2:", round(model.score(X, Y), 3))

training R2: 0.953


In [11]:
from qsarkit.explainability import PermutationImportance

importance = PermutationImportance(n_repeats=5, random_state=0).fit(model, X, Y)
top = np.argsort(-importance.importances_mean_)[:8]
for bit in top:
    print(f"  bit {bit:4}  importance {importance.importances_mean_[bit]:.4f} "
          f"+/- {importance.importances_std_[bit]:.4f}")

  bit  389  importance 0.0300 +/- 0.0045
  bit  378  importance 0.0255 +/- 0.0074
  bit  106  importance 0.0246 +/- 0.0047
  bit  456  importance 0.0236 +/- 0.0045
  bit  304  importance 0.0228 +/- 0.0086
  bit  114  importance 0.0225 +/- 0.0044
  bit   49  importance 0.0220 +/- 0.0079
  bit   46  importance 0.0207 +/- 0.0082


Bit indices are not an explanation a chemist can act on. Mapping the model
back onto atoms is.

In [12]:
from qsarkit.explainability import AtomicContributionMap

explainer = AtomicContributionMap(model, fingerprint)
target = 2                       # the 4-chlorobenzoic acid outlier
result = explainer.explain(mols[target])

print(SMILES[target], " predicted", round(float(result.prediction), 2),
      " actual", Y[target])
for atom, weight in zip(mols[target].GetAtoms(), result.weights):
    print(f"  atom {atom.GetIdx():2} {atom.GetSymbol():2}  {weight:+.4f}")

OC(=O)c1ccc(Cl)cc1  predicted 7.02  actual 7.8
  atom  0 O   -0.0828
  atom  1 C   -0.0345
  atom  2 O   -0.2305
  atom  3 C   +0.0091
  atom  4 C   +0.3245
  atom  5 C   +0.7779
  atom  6 C   +0.8716
  atom  7 Cl  +1.3522
  atom  8 C   +0.7779
  atom  9 C   +0.3245


These weights are what RDKit's similarity maps draw, and they are chemically
sensible: the chlorine carries by far the largest positive contribution
(+1.35), and the carboxylic acid oxygens are negative.

Note the symmetry — atoms 4 and 9 share a weight, as do 5 and 8. Those are
symmetry-equivalent positions on the ring, so masking either one removes the
same environments. Identical weights on equivalent atoms is a correctness
check, not a coincidence.

In [13]:
from qsarkit.explainability import FragmentContributionAnalyzer

# Aggregates atom-level attributions over chemically meaningful groups,
# one molecule at a time.
fragments = FragmentContributionAnalyzer(model, fingerprint)
fragments.analyze(mols[target])

,fragment,n_atoms,contribution,mean_contribution
0,Clc1ccccc1,7,4.4377,0.633957
1,O=CO,3,-0.3478,-0.115933


SHAP and LIME are available with the `explainability` extra:

```python
from qsarkit.explainability import LIMEExplainer, SHAPExplainer

shap_values = SHAPExplainer(model).explain(X)      # pip install qsarkit-learn[explainability]
```

`TreeExplainer` is exact and fast for tree ensembles; `KernelExplainer` is
model-agnostic and slow enough that you will want to subsample.

## The report

Everything above, assembled into the document a reviewer reads.

In [14]:
from qsarkit.model_selection import RandomSplitter
from qsarkit.metrics import qsar_regression_report
from qsarkit.reporting import (
    QSARReport, plot_predicted_vs_observed, plot_residuals)

train, test = next(RandomSplitter(test_size=0.25, random_state=0).split(X, Y))
final = QSARRegressor("rf", random_state=0).fit(X[train], Y[train])
predictions = final.predict(X[test])

figure = plot_predicted_vs_observed(Y[test], predictions,
                                    title="Held-out predictions")
figure

In [15]:
plot_residuals(Y[test], predictions)

In [16]:
from qsarkit.applicability import LeverageAD
from qsarkit.reporting import plot_williams

leverage = LeverageAD().fit(X[train]).score_samples(X[test])
plot_williams(leverage, Y[test], predictions)

In [17]:
report = (
    QSARReport(title="Benzoic acid series — pIC50 model",
               author="qsarkit example", endpoint="pIC50")
    .add_dataset_section(n_compounds=len(mols), n_train=len(train),
                         n_test=len(test),
                         source="Synthetic example series")
    .add_model_section(final, descriptors="Morgan ECFP4, 512 bits",
                       hyperparameters={"n_estimators": 100})
    .add_validation_section(qsar_regression_report(Y[test], predictions))
    .add_section("SAR analysis",
                 content={"n_cliffs": cliffs["n_cliffs"],
                          "cliff_ratio": round(cliffs["cliff_ratio"], 4),
                          "SARI": round(scores["sari"], 3)},
                 text="One compound (4-chlorobenzoic acid) accounts for every "
                      "detected cliff and should be checked against the assay "
                      "records before the model is used.")
)
print(report.to_markdown()[:1200])

# Benzoic acid series — pIC50 model

**Endpoint:** pIC50  
**Author:** qsarkit example  
**Created:** 2026-09-09T15:50:46+00:00  

## Dataset

| Property | Value |
|---|---|
| n_compounds | 24 |
| n_train | 18 |
| n_test | 6 |
| source | Synthetic example series |

## Algorithm

| Property | Value |
|---|---|
| algorithm | QSARRegressor |
| descriptors | Morgan ECFP4, 512 bits |
| param.n_estimators | 100 |

## Validation

| Property | Value |
|---|---|
| r2 | 0.8213 |
| rmse | 0.4755 |
| mae | 0.4548 |
| ccc | 0.8707 |
| q2_f2 | 0.8213 |
| average_r2m | 0.4868 |
| delta_r2m | 0.2595 |
| golbraikh_tropsha | {'q2': None, 'r2': 0.952090356733883, 'r0_squared': 0.5614635012622393, 'r0_prime_squared': 0.827873793753361, 'k': 0.9824999317988035, 'k_prime': 1.013277392250615, 'delta_r0_squared': 0.2664102924911217, 'criterion_1_q2': None, 'criterion_2_r2': True, 'criterion_3_r0': False, 'criterion_4_slope': True, 'criterion_5_delta_r0': True, 'q2_available': False, 'passed': False} |

## SAR

## OECD reporting

`OECDReportBuilder` structures the same material around the five validation
principles — and, more usefully, refuses to be quiet about the ones you
skipped.

In [18]:
from qsarkit.reporting import OECDReportBuilder

builder = OECDReportBuilder(title="QMRF — benzoic acid pIC50 model",
                            endpoint="pIC50")
for number, name in builder.PRINCIPLES:
    print(f"  {number}. {name}")

  1. A defined endpoint
  2. An unambiguous algorithm
  3. A defined domain of applicability
  4. Appropriate measures of goodness-of-fit, robustness and predictivity
  5. A mechanistic interpretation, if possible


In [19]:
builder.add_evidence(1, True, {
    "endpoint": "pIC50, synthetic example assay",
    "units": "-log10(M)",
})
builder.add_evidence(2, True, {
    "algorithm": "Random forest on Morgan ECFP4 (512 bits)",
    "software": "qsarkit",
})
builder.add_evidence(4, True, qsar_regression_report(Y[test], predictions))

print("still unaddressed:", builder.unaddressed)

still unaddressed: [3, 5]


Principles 3 and 5 are outstanding. A submission fails review over a
principle nobody noticed was missing, so the builder tracks them explicitly
rather than letting an omission look like an absence of problems.

In [20]:
from qsarkit.applicability import ADAnalyzer, TanimotoSimilarityAD

analyzer = ADAnalyzer(TanimotoSimilarityAD(threshold=0.35)).fit(X[train])
ad = analyzer.report(X[test], Y[test], predictions)

builder.add_evidence(3, True, {
    "method": "Tanimoto similarity to nearest training compound, threshold 0.35",
    "coverage": ad["coverage"],
    "rmse_ratio": round(ad["rmse_ratio"], 3) if np.isfinite(ad["rmse_ratio"]) else None,
})
builder.add_evidence(5, True, {
    "interpretation": "Per-atom contribution maps and matched molecular pair "
                      "analysis; activity is driven by the 4-position "
                      "substituent on a benzoic acid core.",
})
print("still unaddressed:", builder.unaddressed)

still unaddressed: []


In [21]:
qmrf = builder.build()
print(qmrf.to_markdown()[:1500])

# QMRF — benzoic acid pIC50 model

**Endpoint:** pIC50  
**Created:** 2026-09-09T15:50:46+00:00  

## OECD validation principles

This report follows the QSAR Model Reporting Format (QMRF) and documents the model against the five OECD validation principles. A principle marked not addressed is an explicit gap, not an omission.

| Property | Value |
|---|---|
| Principle 1 | addressed |
| Principle 2 | addressed |
| Principle 3 | addressed |
| Principle 4 | addressed |
| Principle 5 | addressed |

## Principle 1: A defined endpoint

| Property | Value |
|---|---|
| status | addressed |
| endpoint | pIC50, synthetic example assay |
| units | -log10(M) |

## Principle 2: An unambiguous algorithm

| Property | Value |
|---|---|
| status | addressed |
| algorithm | Random forest on Morgan ECFP4 (512 bits) |
| software | qsarkit |

## Principle 3: A defined domain of applicability

| Property | Value |
|---|---|
| status | addressed |
| method | Tanimoto similarity to nearest training compoun

The report renders three ways — Markdown for a repository, HTML for
sharing, JSON for a downstream system — from the same object.

In [22]:
print("HTML:", len(qmrf.to_html()), "characters")
print("JSON keys:", sorted(qmrf.to_dict()))

HTML: 3840 characters
JSON keys: ['author', 'created', 'endpoint', 'sections', 'title']


## What this series showed

1. Curation is not tidying; misaligned labels and mixed units are silent
   and fatal.
2. The representation decides more than the model does.
3. **The split is the experiment.** A random split reported Q² = 0.82 on
   data where a scaffold split reports −0.89.
4. An applicability domain that marks everything in-domain is telling you
   about your split, not your model.
5. One outlier compound accounted for every activity cliff — worth finding
   before fitting anything, not after.